In [2]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error


# 1. Load dataset
dataset = pd.read_csv("Preplacementdata.csv")

dataset=dataset.drop('sl_no',axis=1)

dataset

# 2. Convert categorical columns
df2 = pd.get_dummies(dataset, drop_first=True, dtype=int)

# 3. Separate independent and dependent variables
indep_X = df2.drop('salary', axis=1)
dep_Y = df2['salary']


# 4. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    indep_X, dep_Y,
    test_size=0.25,
    random_state=0
)


# 5. Feature Scaling
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


# 6. RFE algorithms
rfe_models = {
    'Linear Regression': LinearRegression(),
    'SVM Linear': SVR(kernel='linear'),
    'Decision Tree': DecisionTreeRegressor(random_state=0),
    'Random Forest': RandomForestRegressor(n_estimators=10, random_state=0)
}


# 7. Regression algorithms
regression_models = {
    'Linear Regression': LinearRegression(),
    'SVM Linear': SVR(kernel='linear'),
    'SVM Non-Linear': SVR(kernel='rbf'),
    'Decision Tree': DecisionTreeRegressor(random_state=0),
    'Random Forest': RandomForestRegressor(n_estimators=10, random_state=0)
}


# 8. Create empty DataFrames for comparison
feature_df = pd.DataFrame(
    columns=['RFE Algorithm', 'Selected Features']
)

r2_df = pd.DataFrame(
    columns=['RFE Algorithm', 'Linear Regression',
             'SVM Linear', 'SVM Non-Linear',
             'Decision Tree', 'Random Forest']
)

mse_df = pd.DataFrame(
    columns=['RFE Algorithm', 'Linear Regression',
             'SVM Linear', 'SVM Non-Linear',
             'Decision Tree', 'Random Forest']
)


# 9. Apply RFE using different algorithms
for rfe_name, rfe_model in rfe_models.items():

    print("\nRFE Algorithm:", rfe_name)

    rfe = RFE(
        estimator=rfe_model,
        n_features_to_select=3
    )

    rfe.fit(X_train, y_train)

    # Get selected features
    selected_features = indep_X.columns[rfe.support_]

    # Store selected features in DataFrame
    feature_df.loc[len(feature_df)] = [
        rfe_name, list(selected_features)
    ]

    print("Selected Features:", list(selected_features))


    # Transform training and testing data
    X_train_selected = rfe.transform(X_train)
    X_test_selected = rfe.transform(X_test)


    # Create empty dictionaries for current RFE results
    r2_result = {'RFE Algorithm': rfe_name}
    mse_result = {'RFE Algorithm': rfe_name}


    # 10. Train and compare regression algorithms
    for model_name, model in regression_models.items():

        model.fit(X_train_selected, y_train)

        y_pred = model.predict(X_test_selected)

        # Calculate R2
        r2 = r2_score(y_test, y_pred)

        # Calculate MSE
        mse = mean_squared_error(y_test, y_pred)

        # Store comparison results
        r2_result[model_name] = r2
        mse_result[model_name] = mse


    # Add current comparison results to DataFrames
    r2_df.loc[len(r2_df)] = r2_result
    mse_df.loc[len(mse_df)] = mse_result


# 11. Display all comparison DataFrames
print("\n1. FEATURE SELECTION COMPARISON")
print(feature_df.to_string(index=False))


print("\n2. R2 SCORE COMPARISON")
print(r2_df.to_string(index=False))


print("\n3. MSE COMPARISON")
print(mse_df.to_string(index=False))


# 12. Find the best combination
best_index = r2_df[[
    'Linear Regression',
    'SVM Linear',
    'SVM Non-Linear',
    'Decision Tree',
    'Random Forest'
]].stack().idxmax()

best_rfe = r2_df.loc[best_index[0], 'RFE Algorithm']

best_regression = best_index[1]

best_r2 = r2_df.loc[best_index[0], best_regression]

print("\nBEST COMBINATION")
print("RFE Algorithm:", best_rfe)
print("Regression Algorithm:", best_regression)
print("R2 Score:", best_r2)



RFE Algorithm: Linear Regression
Selected Features: ['mba_p', 'specialisation_Mkt&HR', 'status_Placed']

RFE Algorithm: SVM Linear
Selected Features: ['ssc_p', 'hsc_p', 'status_Placed']

RFE Algorithm: Decision Tree
Selected Features: ['ssc_p', 'mba_p', 'status_Placed']

RFE Algorithm: Random Forest
Selected Features: ['ssc_p', 'hsc_p', 'status_Placed']

1. FEATURE SELECTION COMPARISON
    RFE Algorithm                             Selected Features
Linear Regression [mba_p, specialisation_Mkt&HR, status_Placed]
       SVM Linear                 [ssc_p, hsc_p, status_Placed]
    Decision Tree                 [ssc_p, mba_p, status_Placed]
    Random Forest                 [ssc_p, hsc_p, status_Placed]

2. R2 SCORE COMPARISON
    RFE Algorithm  Linear Regression  SVM Linear  SVM Non-Linear  Decision Tree  Random Forest
Linear Regression           0.844674   -0.237211       -0.239329      -0.267782       0.542405
       SVM Linear           0.864275   -0.235380       -0.239260       0.726